# 🎙️ P2 — Speech AI Engineer (v2 — intégration P1)
## Speech-to-Retrieval System (SRS) — INPT Project

**Objectif** : Transformer des fichiers audio en embeddings (dim=768)  
**Nouveauté v2** : Intégration directe des outputs de P1 (`audio_manifest.csv`, `pairs.csv`)

---
### Plan
1. Installation & Setup
2. Chargement des données de P1
3. Modèle Wav2Vec2
4. Fonction `speech_to_embedding()`
5. Batch encoding de tous les audios P1
6. Sauvegarde `audio_embeddings.npy` + `audio_embeddings_index.csv`
7. Caching & optimisation
8. SpeechEncoder class (pour P3)
9. Export ONNX (pour P4)
10. Tests système


In [ ]:
!pip install -q transformers accelerate torchaudio librosa soundfile onnx onnxruntime joblib tqdm mlflow
print('✅ Installation OK')

## ✅ CELLULE 2 — Imports & Config

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import soundfile as sf
import joblib
from pathlib import Path
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'🎮  GPU    : {torch.cuda.get_device_name(0)}')

CONFIG = {
    'model_name'       : 'facebook/wav2vec2-base-960h',
    # Pour multilingue (Darija/FR) : 'facebook/wav2vec2-xls-r-300m'
    'sample_rate'      : 16000,
    'embedding_dim'    : 768,
    'pooling_strategy' : 'mean',
    'batch_size'       : 8,
    'min_duration'     : 1.0,
    'max_duration'     : 20.0,

    # ── Paths issus de P1 ──────────────────────────────────
    'p1_audio_dir'     : './data/audio_clean/',        # audios propres de P1
    'p1_manifest'      : './data/output/audio_manifest.csv', # index P1
    'p1_pairs'         : './data/output/pairs.csv',   # paires P1
    'p1_pairs_train'   : './data/output/pairs_train.csv',
    'p1_pairs_val'     : './data/output/pairs_val.csv',
    'p1_pairs_test'    : './data/output/pairs_test.csv',

    # ── Outputs P2 ─────────────────────────────────────────
    'embeddings_dir'   : './data/embeddings/',
    'embeddings_matrix': './data/embeddings/audio_embeddings.npy',
    'embeddings_index' : './data/embeddings/audio_embeddings_index.csv',
    'cache_file'       : './data/embeddings/cache.pkl',
    'onnx_path'        : './models/speech_encoder.onnx',
}

for p in [CONFIG['embeddings_dir'], './models']:
    os.makedirs(p, exist_ok=True)

print('\n⚙️  Configuration chargée')

## ✅ CELLULE 3 — Charger les données de P1

In [ ]:
# ════════════════════════════════════════════════════════════
# CHARGEMENT DES OUTPUTS DE P1
# ════════════════════════════════════════════════════════════

def load_p1_data(config: dict) -> tuple:
    """
    Charge les fichiers produits par P1.

    Returns:
        (manifest_df, pairs_df, train_df, val_df, test_df)
    """
    loaded = {}

    for name, path in [
        ('manifest', config['p1_manifest']),
        ('pairs',    config['p1_pairs']),
        ('train',    config['p1_pairs_train']),
        ('val',      config['p1_pairs_val']),
        ('test',     config['p1_pairs_test']),
    ]:
        if os.path.exists(path):
            loaded[name] = pd.read_csv(path)
            print(f'   ✅ {name:10s} : {len(loaded[name]):5d} lignes  ← {path}')
        else:
            print(f'   ⚠️  {name:10s} : fichier introuvable → {path}')
            print(f'       (lancer P1 d\'abord ou vérifier le chemin)')
            loaded[name] = pd.DataFrame()

    return loaded


print('📂 Chargement données P1...')
p1_data = load_p1_data(CONFIG)

manifest_df = p1_data['manifest']
pairs_df    = p1_data['pairs']
train_df    = p1_data['train']
val_df      = p1_data['val']
test_df     = p1_data['test']

if not manifest_df.empty:
    print(f'\n📊 Dataset P1 :')
    print(f'   Fichiers audio : {len(manifest_df)}')
    print(f'   Durée totale   : {manifest_df["duration"].sum()/3600:.2f}h')
    print(f'   Sources        : {manifest_df["source"].value_counts().to_dict()}')
    print(manifest_df.head(3))
else:
    print('\n⚠️  Données P1 non trouvées — mode test avec audio synthétique')

## ✅ CELLULE 4 — Chargement modèle Wav2Vec2

In [ ]:
print(f"⏳ Chargement : {CONFIG['model_name']}")

processor = Wav2Vec2Processor.from_pretrained(CONFIG['model_name'])
model     = Wav2Vec2Model.from_pretrained(CONFIG['model_name']).to(DEVICE).eval()

total  = sum(p.numel() for p in model.parameters())
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'✅ Modèle chargé sur {DEVICE}')
print(f'   Paramètres : {total:,}  (gelés : {frozen:,})')

## ✅ CELLULE 5 — Preprocessing & speech_to_embedding()

In [ ]:
def load_audio(path: str) -> np.ndarray | None:
    """Charge un fichier audio → numpy 16kHz mono normalisé."""
    try:
        wav, _ = librosa.load(path, sr=CONFIG['sample_rate'], mono=True)
        dur = len(wav) / CONFIG['sample_rate']
        if dur < CONFIG['min_duration']:  return None
        if dur > CONFIG['max_duration']:  wav = wav[:int(CONFIG['max_duration'] * CONFIG['sample_rate'])]
        peak = np.abs(wav).max()
        return wav / peak if peak > 0 else wav
    except:
        return None


def speech_to_embedding(audio_input,
                         normalize: bool = True) -> np.ndarray | None:
    """
    Convertit un fichier audio en vecteur d'embedding (768,).

    Args:
        audio_input : str (chemin .wav) ou np.ndarray (waveform)
        normalize   : normalisation L2 (recommandé pour cosine similarity)

    Returns:
        np.ndarray (768,) ou None si erreur
    """
    # 1. Chargement
    wav = load_audio(audio_input) if isinstance(audio_input, str) else audio_input
    if wav is None: return None

    # 2. Preprocessing
    inp = processor(wav, sampling_rate=CONFIG['sample_rate'],
                    return_tensors='pt', padding=True).input_values.to(DEVICE)

    # 3. Forward
    with torch.no_grad():
        hidden = model(inp).last_hidden_state  # [1, T, 768]

    # 4. Mean pooling
    emb = hidden.mean(dim=1).squeeze(0).cpu().numpy()  # (768,)

    # 5. Normalisation L2
    if normalize:
        norm = np.linalg.norm(emb)
        if norm > 0: emb = emb / norm

    return emb


print('✅ Fonctions définies : load_audio(), speech_to_embedding()')

## ✅ CELLULE 6 — Batch encoding COMPLET des audios de P1

In [ ]:
# ════════════════════════════════════════════════════════════
# BATCH ENCODING — Encoder TOUS les audios de P1
# Produit : audio_embeddings.npy + audio_embeddings_index.csv
# ════════════════════════════════════════════════════════════

def encode_all_p1_audios(manifest_df: pd.DataFrame,
                           config: dict) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Encode tous les fichiers audio du manifest P1.

    Returns:
        embeddings_matrix : np.ndarray [N, 768]
        index_df          : DataFrame avec audio_id, filepath, embedding_row
    """
    if manifest_df.empty:
        print('⚠️  manifest_df vide — génération de données synthétiques pour test')
        N = 50
        embeddings = np.random.randn(N, 768).astype(np.float32)
        embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
        idx = pd.DataFrame({
            'audio_id'     : [f'test_{i:04d}' for i in range(N)],
            'filename'     : [f'test_{i:04d}.wav' for i in range(N)],
            'embedding_row': list(range(N)),
            'status'       : 'synthetic',
        })
        return embeddings, idx

    audio_files   = manifest_df['filepath'].tolist()
    audio_ids     = manifest_df['audio_id'].tolist()
    batch_size    = config['batch_size']

    all_embeddings, index_rows = [], []
    row_counter = 0

    print(f'🎙️  Encoding {len(audio_files)} fichiers audio (batch={batch_size})...')

    for i in tqdm(range(0, len(audio_files), batch_size), desc='Batch encoding'):
        batch_paths = audio_files[i:i+batch_size]
        batch_ids   = audio_ids[i:i+batch_size]

        waveforms, valid_ids, valid_paths = [], [], []

        for fpath, fid in zip(batch_paths, batch_ids):
            wav = load_audio(fpath)
            if wav is not None:
                waveforms.append(wav)
                valid_ids.append(fid)
                valid_paths.append(fpath)

        if not waveforms:
            continue

        try:
            inp = processor(
                waveforms,
                sampling_rate=config['sample_rate'],
                return_tensors='pt',
                padding=True
            ).input_values.to(DEVICE)

            with torch.no_grad():
                hidden = model(inp).last_hidden_state  # [B, T, 768]
            embs = hidden.mean(dim=1).cpu().numpy()     # [B, 768]

            # Normalisation L2
            norms = np.linalg.norm(embs, axis=1, keepdims=True)
            embs  = embs / np.maximum(norms, 1e-8)

            for fid, fpath, emb in zip(valid_ids, valid_paths, embs):
                all_embeddings.append(emb)
                index_rows.append({
                    'audio_id'     : fid,
                    'filepath'     : fpath,
                    'embedding_row': row_counter,
                    'status'       : 'ok',
                })
                row_counter += 1

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
                print(f'\n⚠️  OOM — réduire batch_size ({batch_size})')

    embeddings_matrix = np.array(all_embeddings, dtype=np.float32)  # [N, 768]
    index_df          = pd.DataFrame(index_rows)

    # Sauvegarder
    np.save(config['embeddings_matrix'], embeddings_matrix)
    index_df.to_csv(config['embeddings_index'], index=False)

    print(f'\n✅ Encoding terminé !')
    print(f'   Matrix : {embeddings_matrix.shape}  → {config["embeddings_matrix"]}')
    print(f'   Index  : {len(index_df)} entrées → {config["embeddings_index"]}')

    return embeddings_matrix, index_df


# Lancer l'encoding
embeddings_matrix, embeddings_index = encode_all_p1_audios(manifest_df, CONFIG)
print(f'\n📐 Shape finale : {embeddings_matrix.shape}  (N audios × 768 dims)')

## ✅ CELLULE 7 — Enrichir pairs.csv avec les embeddings (pour P3)

In [ ]:
# ════════════════════════════════════════════════════════════
# ENRICHISSEMENT pairs.csv
# Ajouter la colonne embedding_row dans pairs.csv pour P3
# ════════════════════════════════════════════════════════════

def enrich_pairs_with_embeddings(pairs_df: pd.DataFrame,
                                   embeddings_index: pd.DataFrame,
                                   output_path: str) -> pd.DataFrame:
    """
    Fusionne pairs.csv avec l'index des embeddings.
    Permet à P3 de retrouver directement le vecteur audio par son index.

    Returns:
        pairs enrichi avec colonne 'embedding_row'
    """
    if pairs_df.empty or embeddings_index.empty:
        print('⚠️  DataFrames vides — skip enrichissement')
        return pairs_df

    # Merge sur audio_id / filename
    emb_map = embeddings_index[['audio_id', 'embedding_row']].copy()

    # Extraire audio_id depuis audio_file (ex: cv_00001.wav → cv_00001)
    if 'audio_file' in pairs_df.columns:
        pairs_df['audio_id'] = pairs_df['audio_file'].apply(lambda x: Path(x).stem)

    enriched = pairs_df.merge(emb_map, on='audio_id', how='left')
    n_matched = enriched['embedding_row'].notna().sum()

    enriched.to_csv(output_path, index=False)
    print(f'✅ pairs enrichi : {n_matched}/{len(enriched)} paires avec embedding_row')
    print(f'   Sauvegardé → {output_path}')

    return enriched


# Enrichir tous les splits
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = f'./data/output/pairs_{split_name}_with_embeddings.csv'
    enrich_pairs_with_embeddings(split_df, embeddings_index, out)

print('\n✅ P3 peut maintenant charger les embeddings avec :')
print('   emb = audio_embeddings[row_idx]  # vecteur (768,)')

## ✅ CELLULE 8 — SpeechEncoder class (pour P3 training)

In [ ]:
class SpeechEncoder(nn.Module):
    """
    Module PyTorch à intégrer dans le Dual Encoder de P3.

    Usage P3:
        speech_enc = SpeechEncoder(frozen=True).to(device)
        emb = speech_enc(input_values)  # [B, 768]
    """
    def __init__(self,
                  model_name: str = 'facebook/wav2vec2-base-960h',
                  frozen: bool = True,
                  pooling: str = 'mean',
                  projection_dim: int = None):
        super().__init__()
        self.wav2vec2  = Wav2Vec2Model.from_pretrained(model_name)
        self.pooling   = pooling

        if frozen:
            for p in self.wav2vec2.parameters():
                p.requires_grad = False

        self.projection = None
        self.out_dim = 768
        if projection_dim:
            self.projection = nn.Sequential(
                nn.Linear(768, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
            self.out_dim = projection_dim

    def forward(self, input_values: torch.Tensor,
                attention_mask: torch.Tensor = None) -> torch.Tensor:
        """ Input: [B, T] → Output: [B, out_dim] normalisé L2 """
        out = self.wav2vec2(input_values, attention_mask=attention_mask)
        h   = out.last_hidden_state  # [B, T, 768]

        if self.pooling == 'mean':
            if attention_mask is not None:
                m = attention_mask.unsqueeze(-1).float()
                emb = (h * m).sum(1) / m.sum(1).clamp(min=1e-8)
            else:
                emb = h.mean(1)
        else:
            emb = h.max(1).values

        if self.projection:
            emb = self.projection(emb)

        return nn.functional.normalize(emb, p=2, dim=-1)


# Test
speech_encoder = SpeechEncoder(model_name=CONFIG['model_name'], frozen=True).to(DEVICE)
dummy = torch.randn(2, 16000 * 3).to(DEVICE)
with torch.no_grad():
    out = speech_encoder(dummy)
print(f'✅ SpeechEncoder : input {dummy.shape} → output {out.shape}')
print(f'   Norme L2 : {out.norm(dim=-1).mean().item():.4f} (≈1.0)')

## ✅ CELLULE 9 — Caching

In [ ]:
class EmbeddingCache:
    def __init__(self, path='./data/embeddings/cache.pkl'):
        self.path  = path
        self._data = joblib.load(path) if os.path.exists(path) else {}
        print(f'💾 Cache : {len(self._data)} entrées')

    def _key(self, p):
        import hashlib
        s = os.stat(p)
        return hashlib.md5(f'{p}_{s.st_size}_{s.st_mtime}'.encode()).hexdigest()

    def get(self, p):  return self._data.get(self._key(p))
    def set(self, p, e): self._data[self._key(p)] = e
    def save(self):
        os.makedirs(os.path.dirname(self.path) or '.', exist_ok=True)
        joblib.dump(self._data, self.path)

cache = EmbeddingCache(CONFIG['cache_file'])
print('✅ Cache initialisé')

## ✅ CELLULE 10 — Export ONNX

In [ ]:
def export_to_onnx(output_path: str) -> bool:
    """Export le speech encoder en ONNX pour P4."""
    import torch.onnx
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)

    dummy = processor(
        np.random.randn(16000 * 3).astype(np.float32),
        sampling_rate=16000, return_tensors='pt'
    ).input_values.to('cpu')

    model_cpu = model.to('cpu').eval()
    try:
        torch.onnx.export(
            model_cpu, dummy, output_path,
            input_names=['input_values'],
            output_names=['last_hidden_state'],
            dynamic_axes={
                'input_values'      : {0: 'batch', 1: 'seq'},
                'last_hidden_state' : {0: 'batch', 1: 'seq'}
            },
            opset_version=14,
            do_constant_folding=True,
        )
        model.to(DEVICE)
        size_mb = os.path.getsize(output_path) / 1e6
        print(f'✅ ONNX exporté : {output_path} ({size_mb:.1f} MB)')
        return True
    except Exception as e:
        print(f'❌ Erreur ONNX : {e}')
        model.to(DEVICE)
        return False

# Décommenter quand prêt :
# export_to_onnx(CONFIG['onnx_path'])
print('✅ export_to_onnx() défini — décommenter pour lancer')

## ✅ CELLULE 11 — Tests système finaux

In [ ]:
print('=' * 55)
print('🧪 TESTS SYSTÈME P2 (v2 — intégration P1)')
print('=' * 55)

passed = 0
tests  = [
    # (nom, condition_fn)
]

# Audio synthétique pour tests
test_wav = np.random.randn(16000 * 3).astype(np.float32)
test_wav /= np.abs(test_wav).max()
sf.write('/tmp/test.wav', test_wav, 16000)

def check(name, cond, detail=''):
    global passed
    print(f"{'✅' if cond else '❌'}  {name}" + (f'  [{detail}]' if detail else ''))
    if cond: passed += 1

e1 = speech_to_embedding('/tmp/test.wav')
check('Output shape = (768,)',     e1.shape == (768,),          f'{e1.shape}')
check('Norme L2 ≈ 1.0',           abs(np.linalg.norm(e1)-1)<0.01, f'{np.linalg.norm(e1):.4f}')
check('Déterminisme',              np.allclose(e1, speech_to_embedding('/tmp/test.wav'), atol=1e-5))
check('Audio numpy → embedding',   speech_to_embedding(test_wav) is not None)
check('SpeechEncoder batch [2,768]', speech_encoder(torch.randn(2,16000*3).to(DEVICE)).shape == (2,768))
check('Embeddings matrix shape',   embeddings_matrix.ndim == 2 and embeddings_matrix.shape[1] == 768,
       f'{embeddings_matrix.shape}')

TOTAL = 6
print('\n' + '='*55)
print(f'   {passed}/{TOTAL} tests passés')
if passed == TOTAL:
    print('   🎉 P2 v2 prête — intégration P1 validée !')
print('='*55)

## 📋 Récap — Outputs P2 v2

| Fichier | Description | Consommé par |
|---|---|---|
| `audio_embeddings.npy` | Matrice [N, 768] de tous les audios | P3, P4 |
| `audio_embeddings_index.csv` | Mapping audio_id → row index | P3, P4 |
| `pairs_*_with_embeddings.csv` | Paires enrichies avec embedding_row | P3 |
| `speech_encoder.py` | Module importable | P3, P4 |
| `speech_encoder.onnx` | Export ONNX | P4 |

### 🔗 Utilisation par P3 (training)
```python
# Charger les embeddings pré-calculés (rapide)
embeddings = np.load('data/embeddings/audio_embeddings.npy')  # [N, 768]
index      = pd.read_csv('data/embeddings/audio_embeddings_index.csv')
pairs      = pd.read_csv('data/output/pairs_train_with_embeddings.csv')

# Récupérer un embedding par row
row = pairs.iloc[0]['embedding_row']
emb = embeddings[int(row)]  # (768,)
```

### 🗓️ Planning
| Jour | Action |
|------|--------|
| **Jour 4** | Validation prétraitement — sync P1/P2/P3 |
| **Jour 6** | Lancer `encode_all_p1_audios()` sur dataset final |
| **Jour 7** | Livrer `audio_embeddings.npy` à P3 |
| **Jour 9** | Export ONNX → P4 |
